# GE2340 Group 26 · Train the lost-and-found matcher

This notebook trains the model used by the local web app. Umbrella photos are no longer in the category list or the training set.

Run all cells from the project folder GE2340. The steps write fresh files into rtifacts/. After training, restart the web app with python app/server.py.


## 1. Setup

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import numpy as np

from src.campus import location_closeness, time_closeness
from src.pipeline import (
    ROOT,
    SEED,
    SOURCES,
    build_queries,
    build_records,
    classification_report,
    dump_json,
    evaluate,
    fit_linear_matcher,
    metadata_arrays,
    oracle_scores,
    print_table,
    random_summary,
    save_figures,
    top_case,
    train_heads,
    tune_fusion,
    write_findings,
)
from src.scoring import clip_fusion, linear_probability
from src.vision import CATEGORIES, COLOURS, DISPLAY_CATEGORY, ClipEncoder, CnnEncoder, text_prompt

print('Project folder:', ROOT)
print('Categories:', CATEGORIES)
assert 'umbrella' not in CATEGORIES, 'umbrella should not be used for training'


## 2. Sample photos and simulate campus reports

In [ ]:
rng = np.random.default_rng(SEED)
art = ROOT / 'artifacts'
thumb_dir = art / 'thumbs'
thumb_dir.mkdir(parents=True, exist_ok=True)

print('Sampling photos and simulating campus reports...')
records = build_records(rng, thumb_dir)
print('Records:', len(records))
print('Colours:', dict(Counter(record['colour'] for record in records)))
print('Categories:', dict(Counter(record['category'] for record in records)))
assert 'umbrella' not in {record['category'] for record in records}

val_queries = build_queries(records, 'val', rng)
test_queries = build_queries(records, 'test', rng)
gallery = [record for record in records if record['split'] != 'train']
gallery.sort(key=lambda record: record['id'])
record_index = {record['id']: index for index, record in enumerate(records)}
train_index = np.array([index for index, record in enumerate(records) if record['split'] == 'train'])
gallery_index = np.array([record_index[record['id']] for record in gallery])
print('Gallery:', len(gallery), 'val queries:', len(val_queries), 'test queries:', len(test_queries))


## 3. Train the frozen-CNN linear heads

In [ ]:
print('Encoding frozen MobileNet features...')
cnn = CnnEncoder()
cnn_features = cnn.encode_images([ROOT / record['source_rel'] for record in records])
scaler, category_model, colour_model, pred_category, pred_colour = train_heads(
    cnn_features, records, train_index, gallery_index
)
for record, category, colour in zip(gallery, pred_category, pred_colour):
    record['pred_category'] = str(category)
    record['pred_colour'] = str(colour)
    record['pred_category_label'] = DISPLAY_CATEGORY[str(category)]
print('Trained category and colour softmax heads')


## 4. Encode CLIP gallery and text prompts

In [ ]:
print('Encoding CLIP gallery and text prompts...')
clip = ClipEncoder()
clip_images = clip.encode_images([ROOT / record['source_rel'] for record in gallery])
text_keys = []
prompts = []
for category in CATEGORIES:
    for colour in COLOURS:
        text_keys.append(f'{colour}|{category}')
        prompts.append(text_prompt(colour, category))
clip_text = clip.encode_texts(prompts)
text_lookup = {key: clip_text[index] for index, key in enumerate(text_keys)}

def visual_of(query: dict) -> np.ndarray:
    vector = text_lookup[f"{query['colour']}|{query['category']}"]
    return clip_images @ vector

print('CLIP:', clip.name)
print('Text prompts:', len(text_keys))


## 5. Fit the linear matcher and tune fusion weights

In [ ]:
coefficients, bias = fit_linear_matcher(
    val_queries, gallery, pred_category, pred_colour, np.random.default_rng(SEED + 1)
)
weights, trials = tune_fusion(val_queries, gallery, visual_of)
print('Fusion weights', {'visual': weights[0], 'location': weights[1], 'time': weights[2]})
print('Linear coefficients', coefficients, 'bias', bias)


## 6. Evaluate and save artifacts for the web app

In [ ]:
def clip_visual_scores(query):
    return visual_of(query)

def clip_fused_scores(query):
    location_close, time_close, _, _ = metadata_arrays(query, gallery)
    fused, _ = clip_fusion(visual_of(query), location_close, time_close, weights)
    return fused

def linear_scores(query):
    _, _, location_far, time_far = metadata_arrays(query, gallery)
    features = np.stack(
        [
            (pred_category == query['category']).astype(np.float32),
            (pred_colour == query['colour']).astype(np.float32),
            location_far,
            time_far,
        ],
        axis=1,
    )
    return linear_probability(features, coefficients, bias)

print('Evaluating...')
splits = {'val': val_queries, 'test': test_queries}
methods = {}
for split_name, queries in splits.items():
    category_counts = dict(Counter(query['category'] for query in queries))
    methods[split_name] = {
        'n_queries': len(queries),
        'gallery_size': len(gallery),
        'methods': {
            'random': random_summary(len(queries), len(gallery), category_counts),
            'metadata_oracle': evaluate(queries, gallery, lambda query: oracle_scores(query, gallery)),
            'clip_visual': evaluate(queries, gallery, clip_visual_scores),
            'clip_fusion': evaluate(queries, gallery, clip_fused_scores),
            'cnn_linear': evaluate(queries, gallery, linear_scores),
        },
    }

classification = classification_report(
    scaler,
    category_model,
    colour_model,
    cnn_features,
    records,
    {
        'val': np.array([record_index[record['id']] for record in gallery if record['split'] == 'val']),
        'test': np.array([record_index[record['id']] for record in gallery if record['split'] == 'test']),
    },
)

successes = []
misses = []
for query in test_queries:
    scores = clip_fused_scores(query)
    case = top_case(query, gallery, scores, 'clip_fusion')
    (successes if case['hit_at_5'] else misses).append(case)
successes.sort(key=lambda case: case['rank'])
misses.sort(key=lambda case: case['rank'], reverse=True)
cases = []
seen_categories = set()
for case in successes:
    if case['query']['category'] in seen_categories:
        continue
    seen_categories.add(case['query']['category'])
    cases.append(case)
    if len(cases) == 3:
        break
if misses:
    cases.append(misses[0])

metrics = {
    'group': {
        'course': 'GE2340',
        'group': '26',
        'title': 'Lost-and-Found Image/Text Matcher',
        'members': [
            {'name': 'So Shing Hin', 'id': '59687520'},
            {'name': 'Tu Kwan Yiu', 'id': '60175669'},
            {'name': 'Chan Kai Shun', 'id': '59696515'},
        ],
    },
    'seed': SEED,
    'dataset': {
        'total_photos': len(records),
        'gallery': len(gallery),
        'by_split': dict(Counter(record['split'] for record in records)),
        'by_category': dict(Counter(record['category'] for record in records)),
        'by_colour': dict(Counter(record['colour'] for record in records)),
        'sources': SOURCES,
    },
    'models': {
        'clip': clip.name,
        'cnn': cnn.name,
        'heads': 'Multinomial logistic regression (softmax) on frozen CNN features',
        'matcher': 'Logistic regression on category match, colour match, location distance, and time difference',
    },
    'fusion_weights': {'visual': weights[0], 'location': weights[1], 'time': weights[2]},
    'weight_search': trials,
    'linear_model': {
        'coefficients': {
            'category_match': float(coefficients[0]),
            'colour_match': float(coefficients[1]),
            'location_distance': float(coefficients[2]),
            'time_difference': float(coefficients[3]),
        },
        'bias': bias,
    },
    'classification': classification,
    'val': methods['val'],
    'test': methods['test'],
    'cases': cases,
    'examples': [
        {
            'label': f"{case['query']['colour'].title()} {DISPLAY_CATEGORY[case['query']['category']]}",
            'query': case['query'],
        }
        for case in cases
        if case['hit_at_5']
    ][:3],
}
metrics['findings'] = write_findings(metrics)

dump_json(art / 'gallery.json', [dict(record) for record in gallery])
dump_json(art / 'queries_val.json', val_queries)
dump_json(art / 'queries_test.json', test_queries)
dump_json(art / 'metrics.json', metrics)
dump_json(art / 'backend.json', {'clip': clip.name, 'cnn': cnn.name, 'text_prompt': 'a photo of a {colour} {category}'})
np.savez_compressed(
    art / 'embeddings.npz',
    ids=np.array([record['id'] for record in gallery]),
    clip=clip_images.astype(np.float32),
    text_keys=np.array(text_keys),
    text_vecs=clip_text.astype(np.float32),
)
save_figures(metrics, art / 'figures')
print_table(metrics)
print('Saved artifacts to', art)
